In [1]:
import os
os.environ['R_HOME'] = '/Users/soeren.becker/micromamba/envs/env_pertpy/lib/R'

In [2]:
import glob
import numpy as np
import pandas as pd
import pertpy as pt
import scanpy as sc
import anndata as ad
import seaborn as sns
import decoupler as dc
import matplotlib.pyplot as plt

from pathlib import Path
from rpy2 import robjects as ro
from rpy2.robjects import numpy2ri, pandas2ri
from rpy2.robjects.conversion import get_conversion, localconverter
from rpy2.robjects.packages import importr
from scipy.sparse import issparse
from tqdm.auto import tqdm

/Users/soeren.becker/micromamba/envs/env_pertpy/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


In [3]:
edger = importr("edgeR")

In [4]:
CELLTYPE = "CD56-bright_NK"

paths = sorted(glob.glob(f"/Users/soeren.becker/repos/HumanCytokineDict/data/pseudobulks/pbulk_{CELLTYPE}_*.h5ad"))
paths = [p for p in paths if (not "Naive" in p) and (not "Memory" in p)]

print(len(paths))

91


In [5]:
def load_data(path):
    pdata = sc.read_h5ad(path)
    cytokine = pdata.obs.cytokine.unique()
    assert len(cytokine) == 1, cytokine
    cytokine = cytokine[0]
    celltype = (Path(path).stem).replace("pbulk_", "").replace("_"+cytokine, "")
    pdata_pbs = sc.read_h5ad(f"/Users/soeren.becker/repos/HumanCytokineDict/data/pseudobulks_unfiltered_pbs_by_well/pbulk_{celltype}_PBS_sum.h5ad")
    return pdata, pdata_pbs, cytokine

def apply_filters_and_concat(pdata, pdata_pbs, frac_cells=0.05, min_cells=10):
    # filter genes
    pdata, pdata_pbs = filter_genes(pdata, pdata_pbs, frac_cells=frac_cells)
    # assert that genes are the same
    assert np.all(pdata.var_names == pdata_pbs.var_names)
    if pdata.shape[1] == 0:
        print(f"{cytokine}: No valid genes for cytokine.")
        return None

    # filter pseudobulk samples
    pdata, pdata_pbs = filter_samples(pdata, pdata_pbs, min_cells=min_cells)
    # assert that samples are paired
    d_pbs = pdata_pbs.obs.donor.unique()
    d_cyto = pdata.obs.donor.unique()
    assert len(np.intersect1d(d_pbs, d_cyto)) == len(d_pbs)
    # check that there are enough samples
    if pdata.shape[0] < 2:
        # we need more than 1 to estimate dispersion
        print(f"{cytokine}: insufficient number of cytokine-treated pseudobulk samples pdata.shape={pdata.shape}.")
        return None
    if pdata_pbs.shape[0] < 2:
        # we need more than 1 to estimate dispersion
        print(f"{cytokine}: insufficient number of PBS-treated pseudobulk samples pdata_pbs.shape={pdata_pbs.shape}.")
        return None
    
    return ad.concat([pdata, pdata_pbs])

def filter_genes(pdata, pdata_pbs, frac_cells=0.05):
    # we keep a gene if it is expressed by at least 5 % of control cells OR 5 % of cytokine-treated cells
    gene_mask = (pdata.var.frac_expressing_cells_total >= frac_cells) | (pdata_pbs.var.frac_expressing_cells_total >= frac_cells)
    valid_genes = pdata.var_names[gene_mask]
    return pdata[:, gene_mask].copy(), pdata_pbs[:, gene_mask].copy()

def filter_samples(pdata, pdata_pbs, min_cells=10):
    # keep donors whose pseudobulks consist of at least 'min_cells' cells in both PBS AND cytokine condition
    valid_donors_1 = pdata.obs.loc[pdata.obs.psbulk_n_cells >= min_cells].donor.values
    valid_donors_2 = pdata_pbs.obs.loc[pdata_pbs.obs.psbulk_n_cells >= min_cells].donor.values
    valid_donors = np.intersect1d(valid_donors_1, valid_donors_2)
    return pdata[pdata.obs.donor.isin(valid_donors)].copy(), pdata_pbs[pdata_pbs.obs.donor.isin(valid_donors)].copy()

def run_de_test(pdata, contrast: str, design=None):
    if design is None:
        design = "~donor+cytokine"
    edgr = pt.tl.EdgeR(pdata, design=design)
    edgr.fit()
    res_df = edgr.test_contrasts(
        edgr.contrast(
            column="cytokine",
            baseline="PBS",
            group_to_compare=contrast,
        )
    )
    return res_df

In [6]:
all_dfs = []

for k, path in enumerate(tqdm(paths)):

    # load and filter
    pdata, pdata_pbs, cytokine = load_data(path)
    if cytokine == "PBS":
        continue

    # apply filters on genes and samples, concat pbs and cytokine
    pdata_both = apply_filters_and_concat(pdata, pdata_pbs, frac_cells=0.05, min_cells=10)
    if pdata_both is None:
        # insufficient number of genes or samples
        continue
    
    # run test
    for pbs_well in sorted(pdata_pbs.obs.bc1_well.unique(), key=lambda x: int(x[1])):
        
        _pdata_both = pdata_both[(pdata_both.obs.cytokine != "PBS") | (pdata_both.obs.bc1_well == pbs_well)]
        
        df = run_de_test(_pdata_both, design="~donor+cytokine", contrast=cytokine)
        df.loc[:, "pbs_well"] = pbs_well
        df.loc[:, "celltype"] = CELLTYPE
        df.loc[:, "cytokine"] = cytokine
        df.loc[:, "num_donors"] = _pdata_both.obs.donor.nunique()
        df.loc[:, "median_num_cells_pbs"] = _pdata_both.obs.loc[_pdata_both.obs.cytokine == "PBS"].psbulk_n_cells.median()
        df.loc[:, "median_num_cells_cytokine"] = _pdata_both.obs.loc[_pdata_both.obs.cytokine != "PBS"].psbulk_n_cells.median()
        df.loc[:, "mean_num_cells_pbs"] = _pdata_both.obs.loc[_pdata_both.obs.cytokine == "PBS"].psbulk_n_cells.mean()
        df.loc[:, "mean_num_cells_cytokine"] = _pdata_both.obs.loc[_pdata_both.obs.cytokine != "PBS"].psbulk_n_cells.mean()
        df.loc[:, "min_num_cells_pbs"] = _pdata_both.obs.loc[_pdata_both.obs.cytokine == "PBS"].psbulk_n_cells.min()
        df.loc[:, "min_num_cells_cytokine"] = _pdata_both.obs.loc[_pdata_both.obs.cytokine != "PBS"].psbulk_n_cells.min()
        df.loc[:, "max_num_cells_pbs"] = _pdata_both.obs.loc[_pdata_both.obs.cytokine == "PBS"].psbulk_n_cells.max()
        df.loc[:, "max_num_cells_cytokine"] = _pdata_both.obs.loc[_pdata_both.obs.cytokine != "PBS"].psbulk_n_cells.max()
        all_dfs.append(df)
        
result_df = pd.concat(all_dfs, axis=0)
result_df.to_csv(f"/Users/soeren.becker/repos/HumanCytokineDict/data/DEGs/DEGs_{CELLTYPE}_separate_pbs_wells.csv")

  0%|                                                                                                                                                                       | 0/91 [00:00<?, ?it/s]

• Calculating NormFactors
• Estimating Dispersions
• Fitting linear model
• Calculating NormFactors
• Estimating Dispersions
• Fitting linear model
• Calculating NormFactors
• Estimating Dispersions
• Fitting linear model
• Calculating NormFactors
• Estimating Dispersions
• Fitting linear model
• Calculating NormFactors
• Estimating Dispersions
• Fitting linear model
• Calculating NormFactors
• Estimating Dispersions
• Fitting linear model


  1%|█▋                                                                                                                                                             | 1/91 [00:20<30:45, 20.51s/it]

• Calculating NormFactors
• Estimating Dispersions
• Fitting linear model
• Calculating NormFactors
• Estimating Dispersions
• Fitting linear model
• Calculating NormFactors
• Estimating Dispersions
• Fitting linear model
• Calculating NormFactors
• Estimating Dispersions
• Fitting linear model
• Calculating NormFactors
• Estimating Dispersions
• Fitting linear model
• Calculating NormFactors
• Estimating Dispersions
• Fitting linear model


  2%|███▍                                                                                                                                                           | 2/91 [00:41<30:44, 20.72s/it]

...

 98%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▌   | 89/91 [34:45<00:39, 19.69s/it]

• Calculating NormFactors
• Estimating Dispersions
• Fitting linear model
• Calculating NormFactors
• Estimating Dispersions
• Fitting linear model
• Calculating NormFactors
• Estimating Dispersions
• Fitting linear model
• Calculating NormFactors
• Estimating Dispersions
• Fitting linear model
• Calculating NormFactors
• Estimating Dispersions
• Fitting linear model
• Calculating NormFactors
• Estimating Dispersions
• Fitting linear model


 99%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▎ | 90/91 [35:04<00:19, 19.38s/it]

• Calculating NormFactors
• Estimating Dispersions
• Fitting linear model
• Calculating NormFactors
• Estimating Dispersions
• Fitting linear model
• Calculating NormFactors
• Estimating Dispersions
• Fitting linear model
• Calculating NormFactors
• Estimating Dispersions
• Fitting linear model
• Calculating NormFactors
• Estimating Dispersions
• Fitting linear model
• Calculating NormFactors
• Estimating Dispersions
• Fitting linear model


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 91/91 [35:23<00:00, 23.33s/it]
